In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, f1_score, classification_report
PROJECT_ROOT = os.path.abspath("..") 
sys.path.append(PROJECT_ROOT)

In [2]:

from dataset.otto_trace import TraceOttoDataSet
from model.trace import TRACE
from utils.SplitData import split_data_Train_Val_Test
from utils.feature_engineering import get_between_features, get_elapsed_feature
from utils.plot_confussion_matrix import plot_confusion_matrix
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



TypeError: Too many arguments for typing.List; actual 2, expected 1

In [6]:
best_checkpoint = torch.load("")

FileNotFoundError: [Errno 2] No such file or directory: ''

In [ ]:
trace_model = TRACE(
    num_embeddings_aid=1855603,
    num_embeddings_event_type=4,
    embedding_dim=32,
    num_classes=1 
)
trace_model = trace_model.to(device)

In [ ]:
trace_model.load_state_dict(best_checkpoint["model_state_dict"])

In [ ]:
dataset_processed  = TraceOttoDataSet(
    file_name='../train.jsonl',
    input_seq_len=64,
    min_timestamps_per_sample=16,
    max_samples=100
)

In [ ]:
_, _, test_loader = split_data_Train_Val_Test(dataset_processed, batch_size=32)

In [ ]:
trace_model.eval()
criterion = nn.BCEWithLogitsLoss()
test_loss = 0
test_correct = 0
test_total = 0
test_accuracy = 0 


all_y_true = []
all_y_pred = []

with torch.no_grad():
    for inputs_test, targets_test in test_loader:
        label_test_task = targets_test["ATC"].unsqueeze(1)
        
        inputs_test = {k: v.to(device)  for k, v in inputs_test.items()}
        
        delta_elapsed = get_elapsed_feature(inputs_test["timestamps"]).to(device)
        delta_between = get_between_features(inputs_test["timestamps"]).to(device)
        
        logit_test = trace_model(
                    inputs_test["aid"],
                    inputs_test["type"],
                    delta_elapsed,
                    delta_between
                )
    
        loss_test = criterion(logit_test, label_test_task.float())
        test_loss += loss_test.item()
        
         #RA1 Debuggin
        probs_test = torch.sigmoid(logit_test)
        preds_test = (probs_test >= 0.5).float()
        test_correct += (preds_test == label_test_task).sum().item()
        test_total += label_test_task.numel()
        
        all_y_true.append(label_test_task.cpu())
        all_y_pred.append(preds_test.cpu())
    
    
        
    test_loss /= len(test_loader)
    test_accuracy = test_correct / max(test_total, 1)
    y_true = torch.cat(all_y_true).numpy().astype(int).ravel()
    y_pred = torch.cat(all_y_pred).numpy().astype(int).ravel()
    confusion_matrix_test = confusion_matrix(y_true, y_pred) # y_true and y_pred
    
    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_accuracy:.4f}")
    print("Confusion Matrix:\n", confusion_matrix_test)
    print(classification_report(y_true, y_pred, target_names=["class 0", "class 1"]))

    
    fig = plot_confusion_matrix(confusion_matrix_test, name_task="PD1", name_classes=["class 0", "class 1"])
    plt.close()

    
    
    